# 02 · Conformers

Sample, compare, deduplicate, pick representatives, search torsions and re-rank with a real energy; each step works
the same on any `Ensemble`, constrained or not. `prune`, `compare` and `representatives` need the `workflow` extra
and `mc` the `search` extra.

In [ ]:
import os
import shutil

import rxembed as rx
import xyzrender

ACID = "OC(=O)CCCCc1ccccc1"  # 5-phenylpentanoic acid: five rotatable bonds

## Sample and compare

`compare(reference, candidate, max_rmsd=)` returns a `Comparison` when two conformers match within `max_rmsd`
angstrom under graph symmetry, and None otherwise, moving neither. Here it finds the duplicates of the lowest
conformer.

In [ ]:
ens = rx.embed(ACID, n=20, seed=1).minimize()
print(ens)

lowest = ens.lowest(1).ids[0]
matches = {cid: ens.compare(lowest, cid, max_rmsd=0.5) for cid in ens.ids if cid != lowest}
print(f"duplicates of conformer {lowest}:", {cid: round(m.rmsd, 2) for cid, m in matches.items() if m is not None})

## Prune

`prune()` drops each conformer within 0.5 angstrom heavy-atom RMSD of a lower one and lists it in `discarded`.
`by="energy"` merges conformers of nearly equal energy without reading coordinates, so it is cheaper but can merge
different shapes.

In [ ]:
by_rmsd = rx.embed(ACID, n=20, seed=1).prune()
by_energy = rx.embed(ACID, n=20, seed=1).prune(by="energy")
print(f"RMSD keeps {by_rmsd.n} of 20; energy keeps {by_energy.n} of 20")
print("dropped by RMSD:", by_rmsd.discarded)

## Representatives

`representatives()` clusters the conformers by their torsions and keeps the lowest of each cluster: a short list to
look at or to pass to an expensive method.

In [ ]:
reps = by_rmsd.representatives()
print(reps)
xyzrender.render(xyzrender.load(reps.mol, ensemble=True))

## Monte Carlo search

`mc()` runs an openconf torsional search, repeatable with `seed=`, and on an unconstrained molecule replaces the
embedded seeds with what it finds. With held atoms or contacts it freezes those atoms and moves only the free
rotors.

In [ ]:
searched = rx.embed(ACID, seed=1).mc(seed=1).prune()
print(searched)
xyzrender.render(xyzrender.load(searched.representatives().mol, ensemble=True))

## A metal complex

`preset="transition_metal"` keeps the coordination sphere and searches the ligands. Here it searches trans
dichlorobis(propylamine)palladium(II), one of the two arrangements `rx.isomers` lists.

In [ ]:
arrangements = rx.isomers("CCCN->[Pd+2](<-[Cl-])(<-[Cl-])<-NCCC", geometry="SPL")
print(arrangements.summary())

palladium = rx.embed(arrangements.select(label="trans"), n=4, seed=1)
palladium.mc(preset="transition_metal", seed=1).prune()
print(palladium)

xyzrender.render(xyzrender.load(palladium.representatives().mol, ensemble=True))

## Real energies (optional)

`score("gxtb")` returns a new `Ensemble` with a g-xTB single point per conformer, a better ranking than UFF's. It
needs an xtb executable, set here by path; the cell skips when none is found.

In [ ]:
XTB = os.path.expanduser("~/bin/xtb")  # your xtb build

if shutil.which(XTB) is None:
    print(f"skipped: no xtb executable at {XTB}")
else:
    os.environ["XTB_EXE"] = XTB
    best = searched.lowest(4)
    scored = best.score("gxtb")
    print("UFF   ", {cid: round(e, 2) for cid, e in best.relative().items()})
    print("g-xTB ", {cid: round(e, 2) for cid, e in scored.relative().items()})